TrustLine — AI Response Quality, Trust & Access Audit
This notebook ingests a log of TrustLine (banking assistant) query/response pairs and:

Cleans and validates the raw data
Analyzes response quality (hallucination rate, confidence distribution)
Detects identity/access-level mismatches (responses that exceed the requesting user's clearance)
Trains a scikit-learn model to flag likely-hallucinated responses
Trains an equivalent PyTorch neural network for the same task
Summarizes findings in an Insights Report, a secure system prompt, and Responsible AI recommendations
Dataset: data/trustline_queries.csv (100 logged interactions).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

try:
    import torch
    import torch.nn as nn
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False
    print("PyTorch not installed in this environment — install with `pip install torch` to run Section 5.")

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_colwidth', 60)

1. Data Ingestion & Cleaning

In [2]:
df_raw = pd.read_csv('data/trustline_queries.csv')
print(df_raw.shape)
df_raw.info()

(100, 10)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Query_ID               100 non-null    object 
 1   User_Role              100 non-null    object 
 2   User_Query             98 non-null     object 
 3   AI_Response            99 non-null     object 
 4   Contains_PII           100 non-null    object 
 5   Confidence_Score       99 non-null     float64
 6   Hallucination_Flag     100 non-null    object 
 7   Accessibility_Format   100 non-null    object 
 8   Access_Level_Required  100 non-null    object 
 9   Response_Source        99 non-null     object 
dtypes: float64(1), object(9)
memory usage: 7.9+ KB
